In [40]:
#| default_exp 30_multihop-datasets

In [33]:
#| export
import pandas as pd, os, json, numpy as np, scipy.sparse as sp, re

from tqdm.auto import tqdm
from typing import Optional, List, Dict

from sugar.core import *

## Helper function

In [34]:
#| export
def load_jsonl(fname):
    data = list()
    with open(fname) as file:
        for line in file: data.append(json.loads(line))
    return data
    

In [35]:
#| export
def load_json(fname:str):
    with open(fname) as f:
        return json.load(f)
        

## `Musique`

In [36]:
#| export
def load_musique_raw(fname:str, title:Optional[List]=None, vocab:Optional[Dict]=None):
    raw_data = load_jsonl(fname)
    
    qry_ids, qry_txt = [], []

    title = [] if title is None else title
    vocab = {} if vocab is None else vocab

    data, indices, indptr = [], [], [0]

    for o in tqdm(raw_data):
        if o["answerable"]:
            qry_ids.append(o["id"])
            qry_txt.append(o["question"])
    
            for lbl in o["paragraphs"]:
                if lbl["paragraph_text"] in vocab:
                    idx = vocab[lbl["paragraph_text"]]
                else:
                    idx = len(vocab)
                    vocab[lbl["paragraph_text"]] = idx
                    title.append(lbl["title"])
    
                if lbl["is_supporting"]:
                    data.append(1.0)
                    indices.append(idx)
                    
            indptr.append(len(indices))
    
    matrix = sp.csr_matrix((data, indices, indptr), shape=(len(indptr) - 1, len(vocab)))
    return qry_ids, qry_txt, title, vocab, matrix
    

In [37]:
#| export
def process_musique(data_dir:str, save_dir:str):
    trn_ids, trn_txt, title, vocab, trn_lbl = load_multihop_raw(f"{data_dir}/musique_ans_v1.0_train.jsonl")
    
    tst_ids, tst_txt, title, vocab, tst_lbl = load_multihop_raw(f"{data_dir}/musique_ans_v1.0_dev.jsonl", 
                                                                title=title, vocab=vocab)
    
    trn_lbl.resize((trn_lbl.shape[0], tst_lbl.shape[1]))

    os.makedirs(f"{save_dir}/raw_data", exist_ok=True)

    save_raw_file(f"{save_dir}/raw_data/train.raw.csv", trn_ids, trn_txt)
    sp.save_npz(f"{save_dir}/trn_X_Y.npz", trn_lbl)

    save_raw_file(f"{save_dir}/raw_data/test.raw.csv", tst_ids, tst_txt)
    sp.save_npz(f"{save_dir}/tst_X_Y.npz", tst_lbl)

    lbl_txt = sorted(vocab, key=lambda x: vocab[x])
    lbl_txt = [f"{p} [SEP] {q}" for p,q in zip(title, lbl_txt)]
    lbl_ids = list(range(len(lbl_txt)))
    save_raw_file(f"{save_dir}/raw_data/label.raw.csv", lbl_ids, lbl_txt)

    return (trn_ids, trn_txt, trn_lbl), (tst_ids, tst_txt, tst_lbl), (lbl_ids, lbl_txt)
    

In [160]:
data_dir = "/Users/suchith720/Projects/data/multihop/musique/original_data"
save_dir = "/Users/suchith720/Projects/data/multihop/musique/XC/"

In [161]:
trn_data, tst_data, lbl_data = process_musique(data_dir, save_dir)

  0%|          | 0/19938 [00:00<?, ?it/s]

  0%|          | 0/2417 [00:00<?, ?it/s]

In [162]:
trn_data[2]

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 46613 stored elements and shape (19938, 101962)>

In [163]:
tst_data[2]

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6404 stored elements and shape (2417, 101962)>

## `2WikiMultiHopQA`

In [38]:
#| export
def load_2wikimultihopqa_raw(fname:str, vocab:Optional[Dict]=None):
    raw_data = load_json(fname)

    qry_ids, qry_txt = [], []

    title = [] if title is None else title
    vocab = {} if vocab is None else vocab
    
    data, indices, indptr = [], [], [0]
    
    for o in tqdm(raw_data):
        qry_ids.append(o["_id"])
        qry_txt.append(o["question"])
    
        supporting_facts = set([f[0] for f in o["supporting_facts"]])
        labels = [(p, f"{p} [SEP] {' '.join(q)}") for p,q in o["context"]]
        
        for fact, lbl in labels:
            idx = vocab.setdefault(lbl, len(vocab))
        
            if fact in supporting_facts:
                data.append(1.0)
                indices.append(idx)
                
        indptr.append(len(indices))
        
    matrix = sp.csr_matrix((data, indices, indptr), shape=(len(indptr) - 1, len(vocab)))
    return qry_ids, qry_txt, vocab, matrix
    

In [39]:
#| export
def process_2wikimultihopqa(data_dir:str, save_dir:str):
    trn_ids, trn_txt, vocab, trn_lbl = load_2wikimultihopqa_raw(f"{data_dir}/train.json")
    
    tst_ids, tst_txt, vocab, tst_lbl = load_2wikimultihopqa_raw(f"{data_dir}/dev.json", vocab=vocab)
    
    trn_lbl.resize((trn_lbl.shape[0], tst_lbl.shape[1]))

    os.makedirs(f"{save_dir}/raw_data", exist_ok=True)

    save_raw_file(f"{save_dir}/raw_data/train.raw.csv", trn_ids, trn_txt)
    sp.save_npz(f"{save_dir}/trn_X_Y.npz", trn_lbl)

    save_raw_file(f"{save_dir}/raw_data/test.raw.csv", tst_ids, tst_txt)
    sp.save_npz(f"{save_dir}/tst_X_Y.npz", tst_lbl)

    lbl_txt = sorted(vocab, key=lambda x: vocab[x])
    lbl_txt = [f"{p} [SEP] {q}" for p,q in zip(title, lbl_txt)]
    lbl_ids = list(range(len(lbl_txt)))
    save_raw_file(f"{save_dir}/raw_data/label.raw.csv", lbl_ids, lbl_txt)

    return (trn_ids, trn_txt, trn_lbl), (tst_ids, tst_txt, tst_lbl), (lbl_ids, lbl_txt)
    

In [170]:
data_dir = "/Users/suchith720/Projects/data/multihop/2wikimultihopqa/original_data/"
save_dir = "/Users/suchith720/Projects/data/multihop/2wikimultihopqa/XC/"

In [ ]:
process_2wikimultihopqa(data_dir, save_dir)

## `Hover`

In [13]:
data_dir = "/Users/suchith720/Projects/data/multihop/hover/original_data/"
save_dir = "/Users/suchith720/Projects/data/multihop/hover/XC/"

In [14]:
!ls /Users/suchith720/Projects/data/multihop/hover/original_data/

hover_dev_release_v1.1.json   hover_train_release_v1.1.json
hover_test_release_v1.1.json


In [28]:
fname = f"{data_dir}/hover_train_release_v1.1.json"

In [29]:
data = load_json(fname)

In [32]:
data[100]

{'uid': 'cb424d8e-0047-4b13-8385-2d6d13af20fd',
 'claim': "A hockey team calls Madison Square Garden it's home. That team, along with the New York Islanders, and the Brooklyn Nets NHL franchise, are popular in the New York metropolitan area. That team, along with the New York Islanders, and the Brooklyn Nets NHL franchise, are popular in the a part of a city.  Coverage of news on Vos Iz Neias? focuses on this part of a city.",
 'supporting_facts': [['1974–75 New York Islanders season', 2],
  ['New York Rangers', 3],
  ['Madison Square Garden', 5],
  ['Vos Iz Neias?', 3]],
 'label': 'NOT_SUPPORTED',
 'num_hops': 4,
 'hpqa_id': '5a820e8855429926c1cdae14'}